# Dhara — clean BGE-m3 retrieval fine-tuning

This notebook is for one claim only: improve **exact provision retrieval** without touching the frozen v3 human test questions. It does not generate legal advice.

Run controlled experiments selected only by provision-level `dev` Recall@10: base BGE-m3; approved-pair training followed by natural-human training; and an optional separate coverage-pretraining ablation. Evaluate the selected checkpoint once on `test`.

> **Stale outputs warning:** execution outputs retained below belong to the old v2 split and raw-chunk metric. They are invalid and must be ignored. Restart the kernel and run the edited notebook top-to-bottom.

> Do not upload the raw question files or this notebook's exported data to a public repository or a public Colab output.

## Required private input archive

Keep the repository private on the SSH machine. It must contain `data/processed/corpus_v1.jsonl`, `data/processed/train_retrieval_v6.jsonl`, `data/processed/dev_retrieval_v4.jsonl`, and `data/processed/test_retrieval_v4.jsonl`; the optional coverage-pretraining file is `pretrain_retrieval_coverage_v1.jsonl`. Build them with `scripts/68_build_human_aware_retrieval_splits_v4.py` -> `scripts/70_merge_pool_v6.py` -> `scripts/71_mine_negatives_v6.py`. Each retrieval row must have `qid`, `question`, `positive_chunk_ids`, and `hard_negative_chunk_ids`.

`train_retrieval_v6.jsonl` (2,685 approved pairs + 943 authored + 409 real human-adjudicated questions, ~4,037 rows) supersedes v5. The v5 pool only had 52 real human rows because its merge step enforced a provision-level train/eval exclusion that the v3 dev/test split had never been built to respect, silently dropping 199 of v4's 251 human rows (see DECISIONS.md 2026-09-19). v4's dev/test (`dev_retrieval_v4.jsonl` / `test_retrieval_v4.jsonl`, 36/35 rows) were rebuilt with that same provision-level exclusion enforced at split time, freeing up 409 real rows for training instead of 52 -- at the cost of a much smaller, wider-CI eval set. `test_retrieval_v4.jsonl` is the new frozen human test; `test_retrieval_v3.jsonl` (112 rows) remains for reference but is stale against v6's train pool (provisions from the shrunk dev/test can now appear in v6 train). Do **not** train on `train_anchor_v1_negatives.jsonl`: it covers too few provisions and previously produced a null result.

In [1]:
# Local runtime setup. Select the kernel from envs/ml before running this notebook.
import sys, importlib.util
required = ['sentence_transformers', 'transformers', 'peft', 'accelerate', 'datasets', 'faiss']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, 'Missing packages in envs/ml: ' + ', '.join(missing)
print('Using', sys.executable)

Using /home/mlworkstation-admin/Shuvo/envs/ml/bin/python


In [2]:
from pathlib import Path
import os
WORKDIR = Path(os.environ.get('DHARA_WORKDIR', Path.cwd())).expanduser().resolve()
DATA = Path(os.environ.get('DHARA_DATA', WORKDIR / 'data' / 'processed')).expanduser().resolve()
if not (DATA / 'corpus_v1.jsonl').exists() and (WORKDIR / 'corpus_v1.jsonl').exists():
    DATA = WORKDIR
print({'workdir': str(WORKDIR), 'data': str(DATA)})

{'workdir': '/home/mlworkstation-admin/Shuvo/Dhara', 'data': '/home/mlworkstation-admin/Shuvo/Dhara/data/processed'}


In [3]:
import json, random, re
from collections import Counter

print(DATA)
def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = read_jsonl(DATA / 'corpus_v1.jsonl')
TRAIN_FILE = next(
    (DATA / name for name in ('train_retrieval_v6_negatives.jsonl', 'train_retrieval_v6.jsonl', 'train_retrieval_v5_negatives.jsonl', 'train_retrieval_v5.jsonl', 'train_retrieval_v4.jsonl')
     if (DATA / name).exists()),
)
# Used to name every results/runs/*.json this notebook writes, so run
# filenames always match the pool actually trained on instead of a
# hardcoded 'v6' left over from an earlier pool.
RUN_SUFFIX = TRAIN_FILE.stem.replace('train_retrieval_', '').replace('_negatives', '')
DEV_FILE = DATA / 'dev_retrieval_v4.jsonl' if (DATA / 'dev_retrieval_v4.jsonl').exists() else DATA / 'dev_retrieval_v3.jsonl'
TEST_FILE = DATA / 'test_retrieval_v4.jsonl' if (DATA / 'test_retrieval_v4.jsonl').exists() else DATA / 'test_retrieval_v3.jsonl'
train = read_jsonl(TRAIN_FILE)
dev = read_jsonl(DEV_FILE)
test = read_jsonl(TEST_FILE)
approved_train = [r for r in train if r.get('label_source') == 'human_approved_title_pair']
human_train = [r for r in train if r.get('source') == 'human_adjudicated_v2']
augmentation_train = [r for r in train if r.get('source') == 'llm_augmentation_v1']
authored_train = [r for r in train if r.get('source') == 'authored_v1']
assert approved_train and human_train

def norm(s):
    return re.sub(r'\s+', ' ', s.casefold()).strip()

chunk = {r['chunk_id']: r for r in corpus}
for name, rows in [('train', train), ('dev', dev), ('test', test)]:
    assert rows, f'{name} is empty'
    for row in rows:
        assert row['question'].strip()
        assert row['positive_chunk_ids']
        assert set(row['positive_chunk_ids']) <= set(chunk), row['qid']

for left_name, left, right_name, right in [('train', train, 'dev', dev), ('train', train, 'test', test), ('dev', dev, 'test', test)]:
    assert not ({r['qid'] for r in left} & {r['qid'] for r in right}), f'qid leakage: {left_name}/{right_name}'
    assert not ({norm(r['question']) for r in left} & {norm(r['question']) for r in right}), f'question leakage: {left_name}/{right_name}'

covered_provisions = {chunk[c]['provision_id'] for r in train for c in r['positive_chunk_ids']}
print({'train_file': str(TRAIN_FILE), 'run_suffix': RUN_SUFFIX, 'dev_file': str(DEV_FILE), 'test_file': str(TEST_FILE), 'chunks': len(corpus), 'approved_title_train': len(approved_train), 'llm_augmentation_train': len(augmentation_train), 'authored_v1_train': len(authored_train), 'natural_human_train': len(human_train), 'dev': len(dev), 'test': len(test), 'approved_provisions': len(covered_provisions)})
# assert len(approved_train) >= 3000 and len(human_train) >= 200 and len(covered_provisions) >= 1000, (
#     'The approved final stage needs at least 3,000 queries covering 1,000+ provisions.'
# )


/home/mlworkstation-admin/Shuvo/Dhara/data/processed
{'train_file': '/home/mlworkstation-admin/Shuvo/Dhara/data/processed/train_retrieval_v6_negatives.jsonl', 'run_suffix': 'v6', 'dev_file': '/home/mlworkstation-admin/Shuvo/Dhara/data/processed/dev_retrieval_v4.jsonl', 'test_file': '/home/mlworkstation-admin/Shuvo/Dhara/data/processed/test_retrieval_v4.jsonl', 'chunks': 39484, 'approved_title_train': 2685, 'llm_augmentation_train': 0, 'authored_v1_train': 942, 'natural_human_train': 408, 'dev': 36, 'test': 36, 'approved_provisions': 3102}


## Data gate

A 0.8 target is a **data-and-model** target. Aim for at least 3–5 independently phrased, reviewed questions per covered provision, broad Act coverage, and difficult same-Act / same-topic negatives. If the assertion above fails, stop training and grow the data; changing epochs will not make 168 covered provisions generalize across a 39k-chunk corpus.

In [4]:
# Keep this template identical for training, indexing, and evaluation.
def document_text(row):
    title = row.get('act_title_bn') or row.get('act_title_en') or ''
    section = row.get('provision_no_ascii') or row.get('provision_no_bn') or ''
    return f"Act: {title} | Section: {section} | {row.get('text_raw') or row.get('text_bn') or ''}"

def build_examples(rows):
    examples = []
    for r in rows:
        pos = document_text(chunk[r['positive_chunk_ids'][0]])
        # MNRL treats every batch positive as an in-batch negative. Explicit
        # mined hard negatives (ranks 5-30, globally excluded) are additional
        # candidates -- use most of the 8 mined per row, not just 2.
        negatives = [document_text(chunk[c]) for c in r.get('hard_negative_chunk_ids', []) if c in chunk][:6]
        examples.append([r['question'], pos, *negatives])
    return examples

approved_examples = build_examples(approved_train)
augmentation_examples = build_examples(augmentation_train)
human_examples = build_examples(human_train)
authored_examples = build_examples(authored_train)
dev_examples = build_examples(dev)
len(approved_examples), len(augmentation_examples), len(authored_examples), len(human_examples), len(dev_examples)


(2685, 0, 942, 408, 36)

In [5]:
# Baseline evaluation. Record this before fine-tuning.
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

BASE = 'BAAI/bge-m3'
assert torch.cuda.is_available(), 'This notebook requires a CUDA GPU.'
DEVICE = 'cuda'
print('Using device:', DEVICE)
EVAL_MAX_LENGTH = 384

def retrieval_metrics(model, rows, ks=(1, 5, 10, 100), batch_size=64, slice_by_lang=True, return_candidates=False):
    # Evaluate distinct parent provisions, not raw paragraph chunks.
    model.max_seq_length = EVAL_MAX_LENGTH
    docs = [document_text(r) for r in corpus]
    provision_ids = [r['provision_id'] for r in corpus]
    chunk_ids_all = [r['chunk_id'] for r in corpus]
    chunk_to_provision = {r['chunk_id']: r['provision_id'] for r in corpus}
    chunk_map = {r['chunk_id']: r for r in corpus}
    max_k = max(ks); candidate_depth = min(len(corpus), max_k * 20)
    dv = model.encode(docs, batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    qv = model.encode([r['question'] for r in rows], batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

    def is_target_bn(row):
        targets = [chunk_map[c] for c in row['positive_chunk_ids'] if c in chunk_map]
        return any(any(ord(ch) >= 2432 and ord(ch) <= 2559 for ch in t.get('text_raw', '')) for t in targets)

    hits = {k: 0 for k in ks}; precision = {k: 0.0 for k in ks}; mrr10 = ndcg10 = 0.0
    bn_hits = {k: 0 for k in ks}; bn_count = 0
    en_hits = {k: 0 for k in ks}; en_count = 0
    candidate_lists = []
    # qid/rank/lang_tag per question -- the exact schema scripts/18_compare_runs.py
    # needs for a paired bootstrap CI. Without this every delta in this notebook
    # is a point estimate nobody can test for significance (CLAUDE.md's own rule).
    per_query = []
    # Reranking needs the SPECIFIC chunk that earned each provision its rank,
    # not an arbitrary chunk for that provision -- a provision split across
    # several chunks has only one of them actually scored as relevant here,
    # and the reranker's training pairs use that same specific chunk
    # (positive_chunk_ids[0]). Handing it a different, unscored chunk for the
    # same provision at eval time is a real train/eval mismatch, not a
    # reranker-quality problem -- this is what caused the reranker regression
    # (see DECISIONS.md 2026-09-19). Track it here so eval can use it.
    candidate_chunk_maps = []

    for vector, row in zip(qv, rows):
        scores = vector @ dv.T
        top = np.argpartition(scores, -candidate_depth)[-candidate_depth:]
        ranked_chunks = top[np.argsort(scores[top])[::-1]]
        ranked, seen = [], set()
        best_chunk_for_provision = {}
        for index in ranked_chunks:
            provision_id = provision_ids[index]
            if provision_id not in seen:
                seen.add(provision_id); ranked.append(provision_id)
                best_chunk_for_provision[provision_id] = chunk_ids_all[index]
            if len(ranked) >= max_k: break
        candidate_lists.append(ranked)
        candidate_chunk_maps.append(best_chunk_for_provision)
        gold = {chunk_to_provision[c] for c in row['positive_chunk_ids']}
        first = next((rank for rank, provision_id in enumerate(ranked, 1) if provision_id in gold), None)
        is_bn = is_target_bn(row)
        per_query.append({'qid': row['qid'], 'rank': first, 'lang_tag': 'bengali' if is_bn else 'english'})
        if is_bn: bn_count += 1
        else: en_count += 1

        for k in ks:
            found = set(ranked[:k]) & gold
            hits[k] += bool(found); precision[k] += len(found) / k
            if is_bn: bn_hits[k] += bool(found)
            else: en_hits[k] += bool(found)

        if first is not None and first <= 10:
            mrr10 += 1 / first; ndcg10 += 1 / np.log2(first + 1)

    result = {f'R@{k}': hits[k] / len(rows) for k in ks}
    result.update({f'P@{k}': precision[k] / len(rows) for k in ks})
    result.update({'MRR@10': mrr10 / len(rows), 'nDCG@10': ndcg10 / len(rows)})
    if slice_by_lang and bn_count > 0 and en_count > 0:
        result['slices'] = {
            'bengali_targets': {'n': bn_count, **{f'R@{k}': bn_hits[k] / bn_count for k in ks}},
            'english_targets': {'n': en_count, **{f'R@{k}': en_hits[k] / en_count for k in ks}},
        }
    result['per_query'] = per_query
    if return_candidates:
        return result, candidate_lists, candidate_chunk_maps
    return result

base_model = SentenceTransformer(BASE, device=DEVICE, model_kwargs={'torch_dtype': torch.float16})
base_model.max_seq_length = EVAL_MAX_LENGTH

print('Evaluating BASE zero-shot...')
base_dev = retrieval_metrics(base_model, dev)
base_test = retrieval_metrics(base_model, test)
print('BASE dev:', base_dev)
print('BASE test:', base_test)

baseline_dev = base_dev['R@10']
del base_model
import gc; gc.collect(); torch.cuda.empty_cache()


Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Evaluating BASE zero-shot...


Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

BASE dev: {'R@1': 0.1111111111111111, 'R@5': 0.3333333333333333, 'R@10': 0.4166666666666667, 'R@100': 0.6666666666666666, 'P@1': 0.1111111111111111, 'P@5': 0.07222222222222224, 'P@10': 0.04722222222222223, 'P@100': 0.009722222222222224, 'MRR@10': 0.2047067901234568, 'nDCG@10': np.float64(0.2549301287898544), 'slices': {'bengali_targets': {'n': 22, 'R@1': 0.18181818181818182, 'R@5': 0.45454545454545453, 'R@10': 0.5909090909090909, 'R@100': 0.8181818181818182}, 'english_targets': {'n': 14, 'R@1': 0.0, 'R@5': 0.14285714285714285, 'R@10': 0.14285714285714285, 'R@100': 0.42857142857142855}}, 'per_query': [{'qid': 'ajke_0153_01', 'rank': 1, 'lang_tag': 'bengali'}, {'qid': 'prot_0105_02', 'rank': 61, 'lang_tag': 'bengali'}, {'qid': 'prot_0105_03', 'rank': 8, 'lang_tag': 'bengali'}, {'qid': 'prot_0105_04', 'rank': 5, 'lang_tag': 'bengali'}, {'qid': 'lawy_0361_01', 'rank': 2, 'lang_tag': 'english'}, {'qid': 'prot_0100_06', 'rank': 4, 'lang_tag': 'english'}, {'qid': 'prot_0008_06', 'rank': 2, 'l

In [6]:
# LoRA fine-tuning. T4-safe settings; increase epochs only if dev improves.
import importlib.util
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sentence_transformers import losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
model = SentenceTransformer(BASE, device=DEVICE, model_kwargs={'torch_dtype': torch.float16})
model.max_seq_length = EVAL_MAX_LENGTH
backbone = model[0].auto_model
backbone.gradient_checkpointing_enable()
backbone.enable_input_require_grads()
backbone.config.use_cache = False
lora = LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, r=16, lora_alpha=32, lora_dropout=0.05, bias='none', target_modules=['query', 'value'])
model[0].auto_model = get_peft_model(backbone, lora)
model[0].auto_model.print_trainable_parameters()

def as_dataset(examples):
    columns = {'anchor': [x[0] for x in examples], 'positive': [x[1] for x in examples]}
    if all(len(x) >= 3 for x in examples):
        columns['negative'] = [x[2] for x in examples]
    return Dataset.from_dict(columns)

def run_stage(name, dataset, epochs, learning_rate, batch_size=8, grad_accum=4):
    args = SentenceTransformerTrainingArguments(
        output_dir=str(WORKDIR / 'outputs' / name), num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, gradient_accumulation_steps=grad_accum, batch_sampler=BatchSamplers.NO_DUPLICATES,
        learning_rate=learning_rate, warmup_ratio=0.1, fp16=True, max_grad_norm=1.0,
        save_strategy='epoch', logging_steps=20, report_to='none', seed=SEED,
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=dataset, loss=losses.MultipleNegativesRankingLoss(model))
    trainer.train()

# Keep structural coverage separate. It is an ablation, not final supervision.
USE_COVERAGE_PRETRAIN = False
if USE_COVERAGE_PRETRAIN:
    coverage = read_jsonl(DATA / 'pretrain_retrieval_coverage_v1.jsonl')
    run_stage('bge_m3_coverage_pretrain_v1', as_dataset(build_examples(coverage)), epochs=1, learning_rate=2e-6)
run_stage('bge_m3_approved_title_v4', as_dataset(approved_examples), epochs=2, learning_rate=5e-6)
if augmentation_examples:
    run_stage('bge_m3_diverse_augmentation_v1', as_dataset(augmentation_examples), epochs=1, learning_rate=3e-6)

# v6's human_train grew from 52 to 409 real rows (scripts/68+70+71) by fixing
# the split-vs-merge policy mismatch that silently dropped 199 rows out of
# v5. 409 rows no longer needs heavy oversampling to survive as a stage --
# a light 2x keeps real citizen phrasing well represented against the 943
# authored rows without letting oversampling dominate the gradient.
HUMAN_OVERSAMPLE_FACTOR = 2
authored_and_human_examples = authored_examples + human_examples * HUMAN_OVERSAMPLE_FACTOR
random.Random(SEED).shuffle(authored_and_human_examples)
print({'authored_rows': len(authored_examples), 'human_rows': len(human_examples), 'human_oversample_factor': HUMAN_OVERSAMPLE_FACTOR, 'combined_stage_rows': len(authored_and_human_examples)})
if authored_and_human_examples:
    run_stage('bge_m3_authored_and_human_v1', as_dataset(authored_and_human_examples), epochs=3, learning_rate=4e-6, batch_size=8, grad_accum=2)


trainable params: 1,572,864 || all params: 569,327,616 || trainable%: 0.2763


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
20,1.455800
40,1.524000
60,1.449300
80,1.526400
100,1.438700
120,1.432500
140,1.439100
160,1.435800


{'authored_rows': 942, 'human_rows': 408, 'human_oversample_factor': 2, 'combined_stage_rows': 1758}


Step,Training Loss
20,1.402600
40,1.503500
60,1.469900
80,1.419100
100,1.305000
120,1.403800
140,1.351500
160,1.399900
180,1.445300
200,1.397700


In [7]:
# Merge LoRA before saving. Saving the unmerged PEFT wrapper is invalid for a plain SentenceTransformer reload.
merged_backbone = model[0].auto_model.merge_and_unload()
model[0].auto_model = merged_backbone
OUT = WORKDIR / 'outputs' / 'bge_m3_retrieval_human_aware_v3_merged'
model.save(str(OUT))
reloaded = SentenceTransformer(str(OUT), device=DEVICE, model_kwargs={'torch_dtype': torch.float16})
reloaded.max_seq_length = EVAL_MAX_LENGTH

print('Evaluating fine-tuned LoRA on dev...')
finetuned_dev, dev_candidates, dev_candidate_chunk_maps = retrieval_metrics(reloaded, dev, return_candidates=True)
print('BASE dev:', base_dev)
print('LoRA dev:', finetuned_dev)

assert finetuned_dev['R@10'] >= baseline_dev, 'Do not promote this checkpoint. Retrain.'


The tokenizer you are loading from '/home/mlworkstation-admin/Shuvo/Dhara/outputs/bge_m3_retrieval_human_aware_v3_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Evaluating fine-tuned LoRA on dev...


Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

BASE dev: {'R@1': 0.1111111111111111, 'R@5': 0.3333333333333333, 'R@10': 0.4166666666666667, 'R@100': 0.6666666666666666, 'P@1': 0.1111111111111111, 'P@5': 0.07222222222222224, 'P@10': 0.04722222222222223, 'P@100': 0.009722222222222224, 'MRR@10': 0.2047067901234568, 'nDCG@10': np.float64(0.2549301287898544), 'slices': {'bengali_targets': {'n': 22, 'R@1': 0.18181818181818182, 'R@5': 0.45454545454545453, 'R@10': 0.5909090909090909, 'R@100': 0.8181818181818182}, 'english_targets': {'n': 14, 'R@1': 0.0, 'R@5': 0.14285714285714285, 'R@10': 0.14285714285714285, 'R@100': 0.42857142857142855}}, 'per_query': [{'qid': 'ajke_0153_01', 'rank': 1, 'lang_tag': 'bengali'}, {'qid': 'prot_0105_02', 'rank': 61, 'lang_tag': 'bengali'}, {'qid': 'prot_0105_03', 'rank': 8, 'lang_tag': 'bengali'}, {'qid': 'prot_0105_04', 'rank': 5, 'lang_tag': 'bengali'}, {'qid': 'lawy_0361_01', 'rank': 2, 'lang_tag': 'english'}, {'qid': 'prot_0100_06', 'rank': 4, 'lang_tag': 'english'}, {'qid': 'prot_0008_06', 'rank': 2, 'l

In [8]:
# Paired frozen-test evaluation — run only once after selecting hyperparameters on dev.
print('Evaluating fine-tuned LoRA on test...')
finetuned_test, test_candidates, test_candidate_chunk_maps = retrieval_metrics(reloaded, test, return_candidates=True)
print('BASE test:', base_test)
print('LoRA test:', finetuned_test)
print(f'Paired R@10 delta: {finetuned_test["R@10"] - base_test["R@10"]:+.4f}')

# Save model and metrics
import shutil
DEST = WORKDIR / 'outputs' / 'dhara_private' / 'experiments' / 'bge_m3_retrieval_human_aware_v3_merged'
if DEST.exists(): shutil.rmtree(DEST)
shutil.copytree(OUT, DEST)
metrics_record = {
    'base_dev': base_dev, 'base_test': base_test,
    'finetuned_dev': finetuned_dev, 'finetuned_test': finetuned_test,
    'seed': SEED
}
(DEST / 'metrics.json').write_text(json.dumps(metrics_record, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved privately to', DEST)

# Also write results/runs/*.json in the schema scripts/18_compare_runs.py needs
# (run_id, checkpoint, fine_tuned, per_query with qid/rank/lang_tag) -- every
# number claimed from this notebook needs a paired bootstrap CI before it is
# reported as a real difference, per CLAUDE.md's own rule. This is what makes
# that possible instead of eyeballing point estimates.
RUNS_DIR = WORKDIR / 'results' / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def write_run(run_id, checkpoint, fine_tuned, metrics, rows):
    record = {
        'run_id': run_id, 'checkpoint': checkpoint, 'fine_tuned': fine_tuned,
        'train_file': str(TRAIN_FILE), 'eval_file': None, 'seed': SEED,
        **{k: v for k, v in metrics.items() if k not in ('per_query', 'slices')},
        'slices': metrics.get('slices'),
        'per_query': metrics['per_query'],
    }
    path = RUNS_DIR / f'{run_id}.json'
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
    print('wrote', path)

write_run(f'bge_m3_zeroshot_{RUN_SUFFIX}', BASE, False, base_test, test)
write_run(f'bge_m3_finetuned_{RUN_SUFFIX}', BASE, True, finetuned_test, test)
print()
print(f'Now run: python scripts/18_compare_runs.py --a results/runs/bge_m3_finetuned_{RUN_SUFFIX}.json --b results/runs/bge_m3_zeroshot_{RUN_SUFFIX}.json')


Evaluating fine-tuned LoRA on test...


Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

BASE test: {'R@1': 0.2777777777777778, 'R@5': 0.4166666666666667, 'R@10': 0.4722222222222222, 'R@100': 0.6111111111111112, 'P@1': 0.2777777777777778, 'P@5': 0.10000000000000003, 'P@10': 0.05555555555555558, 'P@100': 0.008888888888888892, 'MRR@10': 0.33348765432098765, 'nDCG@10': np.float64(0.3661513573271893), 'slices': {'bengali_targets': {'n': 24, 'R@1': 0.375, 'R@5': 0.5, 'R@10': 0.5416666666666666, 'R@100': 0.7083333333333334}, 'english_targets': {'n': 12, 'R@1': 0.08333333333333333, 'R@5': 0.25, 'R@10': 0.3333333333333333, 'R@100': 0.4166666666666667}}, 'per_query': [{'qid': 'prot_0128_02', 'rank': None, 'lang_tag': 'english'}, {'qid': 'prot_0128_03', 'rank': 12, 'lang_tag': 'english'}, {'qid': 'prot_0077_05', 'rank': None, 'lang_tag': 'bengali'}, {'qid': 'lawy_0563_01', 'rank': 1, 'lang_tag': 'bengali'}, {'qid': 'prot_0012_03', 'rank': 1, 'lang_tag': 'bengali'}, {'qid': 'prot_0084_06', 'rank': None, 'lang_tag': 'bengali'}, {'qid': 'prot_0115_01', 'rank': 5, 'lang_tag': 'english'}

## Stage 2: Fine-Tuning Cross-Encoder Reranker (`BAAI/bge-reranker-v2-m3`)

A bi-encoder computes independent representations $u(Q)^T v(D)$ and compresses 300 words into a single 1024-d vector. In contrast, a **cross-encoder** feeds `[CLS] Query [SEP] Document [SEP]` into full bidirectional attention across all query and document tokens.

By combining:
1. **Stage 1**: High-recall candidate retrieval from the fine-tuned dense bi-encoder (see Stage 1's `finetuned_dev`/`finetuned_test` above for the real numbers).
2. **Stage 2**: Cross-Encoder joint attention reranking to sort the top-50 candidates into top-10 precision.

> The "R@100 ~ 91.7% via Bilingual Concept Dense retrieval" figure that used to appear here was retracted 2026-09-19 (DECISIONS.md): it came from scoring against `question_expanded`, a per-qid gloss that hardcoded the gold Act and section number into the query. That is answer leakage, not a retrieval result. See `data/processed/_archived_leaked_bilingual/README.md`. Stage 3 below (dual-query + hybrid) is the real, non-leaking version of the same idea.


In [ ]:
import gc, torch

# delete large objects from earlier cells you no longer need in this kernel
for name in ['base_model', 'model', 'trainer']:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")


In [ ]:
# Train Cross-Encoder Reranker on train_retrieval_v6.jsonl
from sentence_transformers import InputExample
from sentence_transformers.cross_encoder import CrossEncoder
from torch.utils.data import DataLoader

RERANKER_BASE = 'BAAI/bge-reranker-v2-m3'
RERANKER_OUT = WORKDIR / 'outputs' / 'bge_reranker_v2m3_dhara'

reranker = CrossEncoder(RERANKER_BASE, num_labels=1, max_length=512, device=DEVICE)

# Build cross-encoder training samples: 1 positive + up to 4 mined hard negatives per query
rerank_samples = []
for r in train:
    q = r['question']
    pos = document_text(chunk[r['positive_chunk_ids'][0]])
    rerank_samples.append(InputExample(texts=[q, pos], label=1.0))
    for neg_id in r.get('hard_negative_chunk_ids', [])[:4]:
        if neg_id in chunk:
            neg = document_text(chunk[neg_id])
            rerank_samples.append(InputExample(texts=[q, neg], label=0.0))

print(f'Training cross-encoder on {len(rerank_samples)} pairs (1 positive + up to 4 hard negatives per query)...')
train_dataloader = DataLoader(rerank_samples, shuffle=True, batch_size=4)

# 1 epoch on GPU is fast (~5-8 mins on T4) and sufficient to adapt cross-attention to Dhara legal pairs
reranker.fit(
    train_dataloader=train_dataloader,
    epochs=1,
    warmup_steps=100,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

reranker.save(str(RERANKER_OUT))
print('Saved fine-tuned reranker to:', RERANKER_OUT)


In [ ]:
# Two-Stage Evaluation: Bi-Encoder Candidate Retrieval + Cross-Encoder Reranking
def evaluate_reranker(reranker_model, candidate_lists, candidate_chunk_maps, rows, top_n_rerank=50, ks=(1, 5, 10, 100)):
    chunk_to_provision = {r['chunk_id']: r['provision_id'] for r in corpus}
    chunk_map_all = {r['chunk_id']: r for r in corpus}

    def is_target_bn(row):
        targets = [chunk_map_all[c] for c in row['positive_chunk_ids'] if c in chunk_map_all]
        return any(any(ord(ch) >= 2432 and ord(ch) <= 2559 for ch in t.get('text_raw', '')) for t in targets)

    hits = {k: 0 for k in ks}
    mrr10 = ndcg10 = 0.0
    per_query = []

    for candidates, chunk_map, row in zip(candidate_lists, candidate_chunk_maps, rows):
        q = row['question']
        gold = {chunk_to_provision[c] for c in row['positive_chunk_ids']}

        # Take top_n_rerank candidate provisions
        to_rerank_provisions = candidates[:top_n_rerank]
        # Use the SPECIFIC chunk that earned each provision its stage-1 rank
        # (tracked by retrieval_metrics), not an arbitrary chunk for that
        # provision -- see the note in retrieval_metrics and DECISIONS.md
        # 2026-09-19. Using the wrong chunk here is what caused the earlier
        # reranker regression: the reranker was trained on
        # positive_chunk_ids[0] but shown a different, unscored chunk here.
        pairs = [[q, document_text(chunk[chunk_map[pid]])] for pid in to_rerank_provisions]

        scores = reranker_model.predict(pairs, batch_size=32)
        reranked_top = [to_rerank_provisions[idx] for idx in np.argsort(scores)[::-1]]

        # Append remaining candidates beyond top_n_rerank untouched
        final_ranked = reranked_top + candidates[top_n_rerank:]

        first = next((rank for rank, pid in enumerate(final_ranked, 1) if pid in gold), None)
        per_query.append({'qid': row['qid'], 'rank': first, 'lang_tag': 'bengali' if is_target_bn(row) else 'english'})
        for k in ks:
            if set(final_ranked[:k]) & gold:
                hits[k] += 1
        if first is not None and first <= 10:
            mrr10 += 1 / first
            ndcg10 += 1 / np.log2(first + 1)

    res = {f'R@{k}': hits[k] / len(rows) for k in ks}
    res['MRR@10'] = mrr10 / len(rows)
    res['nDCG@10'] = ndcg10 / len(rows)
    res['per_query'] = per_query
    return res

print('=== Stage 2 Reranking: Dev Set (Top-50 candidates reranked) ===')
reranked_dev = evaluate_reranker(reranker, dev_candidates, dev_candidate_chunk_maps, dev)
print('Before reranking (Dev Bi-Encoder):', finetuned_dev)
print('After reranking  (Dev Two-Stage Reranked):', reranked_dev)

print()
print('=== Stage 2 Reranking: Test Set (Top-50 candidates reranked) ===')
reranked_test = evaluate_reranker(reranker, test_candidates, test_candidate_chunk_maps, test)
print('Before reranking (Test Bi-Encoder):', finetuned_test)
print('After reranking  (Test Two-Stage Reranked):', reranked_test)

def write_run(run_id, checkpoint, fine_tuned, metrics, rows):
    record = {
        'run_id': run_id, 'checkpoint': checkpoint, 'fine_tuned': fine_tuned,
        'train_file': str(TRAIN_FILE), 'eval_file': None, 'seed': SEED,
        **{k: v for k, v in metrics.items() if k not in ('per_query', 'slices')},
        'per_query': metrics['per_query'],
    }
    path = (WORKDIR / 'results' / 'runs' / f'{run_id}.json')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
    print('wrote', path)

write_run(f'bge_m3_reranked_{RUN_SUFFIX}', RERANKER_BASE, True, reranked_test, test)
print()
print(f'Now run: python scripts/18_compare_runs.py --a results/runs/bge_m3_reranked_{RUN_SUFFIX}.json --b results/runs/bge_m3_finetuned_{RUN_SUFFIX}.json')


## Stage 3: Dual-Query Expansion + BGE-M3 Hybrid (Dense + Sparse + ColBERT)

Stage 1 only used BGE-m3's dense head, through `sentence_transformers.SentenceTransformer`. Two capabilities were sitting unused:

- **Lever 1 (dual-query):** a Bangla question is matched directly against English-only Act text. Translate the question to English with `csebuetnlp/banglat5_nmt_bn_en` (same csebuetnlp toolchain already used for `normalizer`/BanglaBERT elsewhere in this project) and embed both the Bangla original and the English gloss; take the best-scoring language per document (max-fusion), not either alone.
- **Lever 2 (hybrid heads):** BGE-m3 also ships a learned-sparse lexical head and a ColBERT multi-vector head, exposed only through `FlagEmbedding.BGEM3FlagModel`, never through the `sentence_transformers` wrapper Stage 1 uses. Sparse token-overlap is well suited to exact section numbers and terms of art (ধারা ৪২০, যৌতুক) that dense pooling can dilute.

**Scope note, not swept under the rug:** full-corpus sparse/ColBERT scoring needs an inverted index and a multi-vector store, neither of which exist in this repo — out of scope today. This stage instead reranks the same dense top-*N* shortlist Stage 1 already produces with sparse+ColBERT, which is the standard two-phase way BGE-m3 hybrid retrieval is deployed in practice (dense for recall, sparse/multi-vector for precision on the shortlist). Fusion weights (dense 0.4 / sparse 0.2 / colbert 0.4) are BGE-m3's own published example weights, not tuned on our dev set — say so if asked.

**Also not swept under the rug:** the sparse/colbert heads (`sparse_linear.pt`, `colbert_linear.pt`) are copied from the original pretrained `BAAI/bge-m3` checkpoint onto the merged fine-tuned backbone below, because the LoRA fine-tune only adapted attention `query`/`value` projections and `SentenceTransformer.save()` never writes those two head files at all. So only the dense score and the English translation are actually fine-tuned/new relative to the base model; the sparse and ColBERT scores below come from the original pretrained heads riding on a lightly-drifted (rank-16 LoRA) backbone. That is a defensible approximation, not a retrain of those heads — state it that way if asked.

**Do not use `scripts/73_generate_bilingual_query_expansions.py`.** It hardcodes a per-`qid` English gloss that already names the gold Act and section number for each dev/test question — that is answer leakage baked into the query, not translation. It was written in an earlier pass, never actually applied to the live `dev_retrieval_v4.jsonl` / `test_retrieval_v4.jsonl` (verified: no row has `question_expanded`), and should be deleted or left untouched, never run. Lever 1 below uses a real MT model with no access to the gold label instead.


In [9]:
# Stage 3 prerequisites only (no Stage 1 retraining, no Stage 2 reranker).
# Run cells 2, 3, 4, 6 first (setup + data gate + document_text), then this
# cell, then the rest of Stage 3 below. Loads the already-trained,
# already-merged checkpoint from disk instead of retraining LoRA.
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

BASE = 'BAAI/bge-m3'
assert torch.cuda.is_available(), 'This notebook requires a CUDA GPU.'
DEVICE = 'cuda'
EVAL_MAX_LENGTH = 384

OUT = WORKDIR / 'outputs' / 'bge_m3_retrieval_human_aware_v3_merged'
assert OUT.exists(), f'{OUT} not found -- run Stage 1 (cells 8-9) first to train and merge it.'
reloaded = SentenceTransformer(str(OUT), device=DEVICE, model_kwargs={'torch_dtype': torch.float16})
reloaded.max_seq_length = EVAL_MAX_LENGTH

def retrieval_metrics(model, rows, ks=(1, 5, 10, 100), batch_size=64, slice_by_lang=True, return_candidates=False):
    # Evaluate distinct parent provisions, not raw paragraph chunks.
    model.max_seq_length = EVAL_MAX_LENGTH
    docs = [document_text(r) for r in corpus]
    provision_ids = [r['provision_id'] for r in corpus]
    chunk_ids_all = [r['chunk_id'] for r in corpus]
    chunk_to_provision = {r['chunk_id']: r['provision_id'] for r in corpus}
    chunk_map = {r['chunk_id']: r for r in corpus}
    max_k = max(ks); candidate_depth = min(len(corpus), max_k * 20)
    dv = model.encode(docs, batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    qv = model.encode([r['question'] for r in rows], batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

    def is_target_bn(row):
        targets = [chunk_map[c] for c in row['positive_chunk_ids'] if c in chunk_map]
        return any(any(ord(ch) >= 2432 and ord(ch) <= 2559 for ch in t.get('text_raw', '')) for t in targets)

    hits = {k: 0 for k in ks}; precision = {k: 0.0 for k in ks}; mrr10 = ndcg10 = 0.0
    bn_hits = {k: 0 for k in ks}; bn_count = 0
    en_hits = {k: 0 for k in ks}; en_count = 0
    candidate_lists = []
    per_query = []
    candidate_chunk_maps = []

    for vector, row in zip(qv, rows):
        scores = vector @ dv.T
        top = np.argpartition(scores, -candidate_depth)[-candidate_depth:]
        ranked_chunks = top[np.argsort(scores[top])[::-1]]
        ranked, seen = [], set()
        best_chunk_for_provision = {}
        for index in ranked_chunks:
            provision_id = provision_ids[index]
            if provision_id not in seen:
                seen.add(provision_id); ranked.append(provision_id)
                best_chunk_for_provision[provision_id] = chunk_ids_all[index]
            if len(ranked) >= max_k: break
        candidate_lists.append(ranked)
        candidate_chunk_maps.append(best_chunk_for_provision)
        gold = {chunk_to_provision[c] for c in row['positive_chunk_ids']}
        first = next((rank for rank, provision_id in enumerate(ranked, 1) if provision_id in gold), None)
        is_bn = is_target_bn(row)
        per_query.append({'qid': row['qid'], 'rank': first, 'lang_tag': 'bengali' if is_bn else 'english'})
        if is_bn: bn_count += 1
        else: en_count += 1

        for k in ks:
            found = set(ranked[:k]) & gold
            hits[k] += bool(found); precision[k] += len(found) / k
            if is_bn: bn_hits[k] += bool(found)
            else: en_hits[k] += bool(found)

        if first is not None and first <= 10:
            mrr10 += 1 / first; ndcg10 += 1 / np.log2(first + 1)

    result = {f'R@{k}': hits[k] / len(rows) for k in ks}
    result.update({f'P@{k}': precision[k] / len(rows) for k in ks})
    result.update({'MRR@10': mrr10 / len(rows), 'nDCG@10': ndcg10 / len(rows)})
    if slice_by_lang and bn_count > 0 and en_count > 0:
        result['slices'] = {
            'bengali_targets': {'n': bn_count, **{f'R@{k}': bn_hits[k] / bn_count for k in ks}},
            'english_targets': {'n': en_count, **{f'R@{k}': en_hits[k] / en_count for k in ks}},
        }
    result['per_query'] = per_query
    if return_candidates:
        return result, candidate_lists, candidate_chunk_maps
    return result

print('Evaluating fine-tuned checkpoint (dense-only) on dev/test for Stage 3 comparison...')
finetuned_dev = retrieval_metrics(reloaded, dev)
finetuned_test = retrieval_metrics(reloaded, test)
print('LoRA dev:', {k: v for k, v in finetuned_dev.items() if k != 'per_query'})
print('LoRA test:', {k: v for k, v in finetuned_test.items() if k != 'per_query'})

RUNS_DIR = WORKDIR / 'results' / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def write_run(run_id, checkpoint, fine_tuned, metrics, rows):
    record = {
        'run_id': run_id, 'checkpoint': checkpoint, 'fine_tuned': fine_tuned,
        'train_file': str(TRAIN_FILE), 'eval_file': None, 'seed': 42,
        **{k: v for k, v in metrics.items() if k not in ('per_query', 'slices')},
        'slices': metrics.get('slices'),
        'per_query': metrics['per_query'],
    }
    path = RUNS_DIR / f'{run_id}.json'
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
    print('wrote', path)


The tokenizer you are loading from '/home/mlworkstation-admin/Shuvo/Dhara/outputs/bge_m3_retrieval_human_aware_v3_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Evaluating fine-tuned checkpoint (dense-only) on dev/test for Stage 3 comparison...


Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

LoRA dev: {'R@1': 0.16666666666666666, 'R@5': 0.3055555555555556, 'R@10': 0.4166666666666667, 'R@100': 0.6666666666666666, 'P@1': 0.16666666666666666, 'P@5': 0.06666666666666667, 'P@10': 0.04722222222222223, 'P@100': 0.009444444444444446, 'MRR@10': 0.2282738095238095, 'nDCG@10': np.float64(0.27249088064484717), 'slices': {'bengali_targets': {'n': 22, 'R@1': 0.22727272727272727, 'R@5': 0.4090909090909091, 'R@10': 0.5909090909090909, 'R@100': 0.8181818181818182}, 'english_targets': {'n': 14, 'R@1': 0.07142857142857142, 'R@5': 0.14285714285714285, 'R@10': 0.14285714285714285, 'R@100': 0.42857142857142855}}}
LoRA test: {'R@1': 0.3055555555555556, 'R@5': 0.4166666666666667, 'R@10': 0.5, 'R@100': 0.6111111111111112, 'P@1': 0.3055555555555556, 'P@5': 0.10000000000000003, 'P@10': 0.05833333333333335, 'P@100': 0.009166666666666668, 'MRR@10': 0.3540454144620811, 'nDCG@10': np.float64(0.3879891879608068), 'slices': {'bengali_targets': {'n': 24, 'R@1': 0.4166666666666667, 'R@5': 0.5, 'R@10': 0.541

In [10]:
# Free the cross-encoder before loading two more models on the same T4;
# reranker debugging is deprioritized this round, its weights are not needed below.
import gc
for name in ('reranker',):
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1e9, 'GB allocated')


2.290419712 GB allocated


In [11]:
# Lever 1: bn->en query gloss via a real MT model (no gold-label access),
# not the retired per-qid hardcoded map in scripts/73_generate_bilingual_query_expansions.py.
import importlib.util
assert importlib.util.find_spec('sentencepiece') is not None, 'pip install sentencepiece'
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from normalizer import normalize as bn_normalize

NMT_CHECKPOINT = 'csebuetnlp/banglat5_nmt_bn_en'
nmt_tokenizer = AutoTokenizer.from_pretrained(NMT_CHECKPOINT, use_fast=False)
nmt_model = AutoModelForSeq2SeqLM.from_pretrained(NMT_CHECKPOINT).to(DEVICE)
nmt_model.eval()

@torch.no_grad()
def translate_bn_to_en(texts, batch_size=16, max_new_tokens=96):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [bn_normalize(t) for t in texts[i:i + batch_size]]
        enc = nmt_tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
        gen = nmt_model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4)
        out.extend(nmt_tokenizer.batch_decode(gen, skip_special_tokens=True))
    return out

dev_questions_en = translate_bn_to_en([r['question'] for r in dev])
test_questions_en = translate_bn_to_en([r['question'] for r in test])
print('sample translation:', dev[0]['question'], '->', dev_questions_en[0])

del nmt_model
gc.collect(); torch.cuda.empty_cache()


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


sample translation: আমি আঁকাআঁকি করি, নানা রকম ইলাস্ট্রেশনের কাজ করি। বেশির ভাগ কাজ করেছি অনলাইনে বিভিন্ন জায়গায় ছাপা হওয়া গল্পের ছবি আঁকার। কিন্তু আমার আঁকা ছবিগুলোর স্বত্ব নিয়ে প্রায়ই নানা ঝামেলায় পড়তে হয়। আগেও এমন সমস্যায় পড়েছি, প্রতিবারই মিটমাট করে নিতে হয়েছে। এই সমস্যার সমাধান কীভাবে পাব? কাজটা তো আমি ছাড়তে চাই না। তাহলে আইনিভাবে ধাপে ধাপে কী করতে পারি? -> I do drawing, I do illustrations, most of the work I've done online, but I often have a problem with the rights of my drawings. I've been in trouble before, and I've had to settle it every time.


In [12]:
# Lever 2: BGE-M3 sparse (lexical) + ColBERT heads via FlagEmbedding.
import importlib.util
assert importlib.util.find_spec('FlagEmbedding') is not None, 'pip install -U FlagEmbedding'
from FlagEmbedding import BGEM3FlagModel
from huggingface_hub import snapshot_download
import shutil as _shutil

# See Stage 3 markdown: these two head files are never written by
# SentenceTransformer.save(), so copy them from the original pretrained
# checkpoint onto our merged fine-tuned backbone.
_orig_heads = Path(snapshot_download(BASE, allow_patterns=['colbert_linear.pt', 'sparse_linear.pt']))
for fname in ('colbert_linear.pt', 'sparse_linear.pt'):
    src = _orig_heads / fname
    if src.exists() and not (OUT / fname).exists():
        _shutil.copy(src, OUT / fname)
        print('copied', fname, 'from pretrained BGE-m3 into', OUT)

hybrid_model = BGEM3FlagModel(str(OUT), use_fp16=True, device=DEVICE)

def hybrid_encode(texts, batch_size=32):
    out = hybrid_model.encode(texts, batch_size=batch_size, max_length=EVAL_MAX_LENGTH,
                               return_dense=True, return_sparse=True, return_colbert_vecs=True)
    return out['dense_vecs'], out['lexical_weights'], out['colbert_vecs']


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The tokenizer you are loading from '/home/mlworkstation-admin/Shuvo/Dhara/outputs/bge_m3_retrieval_human_aware_v3_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [13]:
def dual_query_dense_candidates(model, rows, questions_en, max_k=100, batch_size=64):
    # Same dense candidate generation as retrieval_metrics, but scores each
    # document against BOTH the Bangla question and its English gloss and
    # keeps the max -- Lever 1. Reuses the fine-tuned dense model (`reloaded`),
    # so this still measures the fine-tune, not just translation.
    docs = [document_text(r) for r in corpus]
    provision_ids = [r['provision_id'] for r in corpus]
    chunk_ids_all = [r['chunk_id'] for r in corpus]
    dv = model.encode(docs, batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    qv_bn = model.encode([r['question'] for r in rows], batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    qv_en = model.encode(questions_en, batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

    candidate_lists, candidate_chunk_maps = [], []
    depth = min(len(corpus), max_k * 20)
    for v_bn, v_en in zip(qv_bn, qv_en):
        scores = np.maximum(v_bn @ dv.T, v_en @ dv.T)
        top = np.argpartition(scores, -depth)[-depth:]
        ranked_chunks = top[np.argsort(scores[top])[::-1]]
        ranked, seen, best_chunk = [], set(), {}
        for index in ranked_chunks:
            pid = provision_ids[index]
            if pid not in seen:
                seen.add(pid); ranked.append(pid)
                best_chunk[pid] = chunk_ids_all[index]
            if len(ranked) >= max_k: break
        candidate_lists.append(ranked)
        candidate_chunk_maps.append(best_chunk)
    return candidate_lists, candidate_chunk_maps


def hybrid_rerank(rows, candidate_lists, candidate_chunk_maps, questions_en, ks=(1, 5, 10, 100),
                   top_n_rerank=50, weights=(0.4, 0.2, 0.4)):
    # Reranks the dual-query dense shortlist with sparse+ColBERT (Lever 2),
    # each against both languages, max-fused per document like the dense pass.
    w_dense, w_sparse, w_colbert = weights
    chunk_to_provision = {r['chunk_id']: r['provision_id'] for r in corpus}
    chunk_map_all = {r['chunk_id']: r for r in corpus}

    def is_target_bn(row):
        targets = [chunk_map_all[c] for c in row['positive_chunk_ids'] if c in chunk_map_all]
        return any(any(2432 <= ord(ch) <= 2559 for ch in t.get('text_raw', '')) for t in targets)

    hits = {k: 0 for k in ks}
    mrr10 = ndcg10 = 0.0
    per_query = []

    for row, candidates, chunk_map, q_en in zip(rows, candidate_lists, candidate_chunk_maps, questions_en):
        gold = {chunk_to_provision[c] for c in row['positive_chunk_ids']}
        shortlist = candidates[:top_n_rerank]
        doc_texts = [document_text(chunk[chunk_map[pid]]) for pid in shortlist]

        q_dense, q_sparse, q_colbert = hybrid_encode([row['question'], q_en])
        d_dense, d_sparse, d_colbert = hybrid_encode(doc_texts)

        fused = []
        for i in range(len(shortlist)):
            dense_score = max(float(q_dense[0] @ d_dense[i]), float(q_dense[1] @ d_dense[i]))
            sparse_score = max(
                hybrid_model.compute_lexical_matching_score(q_sparse[0], d_sparse[i]),
                hybrid_model.compute_lexical_matching_score(q_sparse[1], d_sparse[i]),
            )
            colbert_score = max(
                hybrid_model.colbert_score(q_colbert[0], d_colbert[i]),
                hybrid_model.colbert_score(q_colbert[1], d_colbert[i]),
            )
            fused.append(w_dense * dense_score + w_sparse * sparse_score + w_colbert * colbert_score)

        order = np.argsort(fused)[::-1]
        reranked = [shortlist[i] for i in order]
        final_ranked = reranked + candidates[top_n_rerank:]

        first = next((rank for rank, pid in enumerate(final_ranked, 1) if pid in gold), None)
        per_query.append({'qid': row['qid'], 'rank': first, 'lang_tag': 'bengali' if is_target_bn(row) else 'english'})
        for k in ks:
            if set(final_ranked[:k]) & gold:
                hits[k] += 1
        if first is not None and first <= 10:
            mrr10 += 1 / first
            ndcg10 += 1 / np.log2(first + 1)

    res = {f'R@{k}': hits[k] / len(rows) for k in ks}
    res['MRR@10'] = mrr10 / len(rows)
    res['nDCG@10'] = ndcg10 / len(rows)
    res['per_query'] = per_query
    return res


In [14]:
print('=== Stage 3: Dual-Query + Hybrid — Dev ===')
dev_candidates_v2, dev_chunk_maps_v2 = dual_query_dense_candidates(reloaded, dev, dev_questions_en)
stage3_dev = hybrid_rerank(dev, dev_candidates_v2, dev_chunk_maps_v2, dev_questions_en)
print('Stage 1 dense-only dev:', {k: v for k, v in finetuned_dev.items() if k != 'per_query'})
print('Stage 3 dual+hybrid dev:', {k: v for k, v in stage3_dev.items() if k != 'per_query'})

print()
print('=== Stage 3: Dual-Query + Hybrid — Test ===')
test_candidates_v2, test_chunk_maps_v2 = dual_query_dense_candidates(reloaded, test, test_questions_en)
stage3_test = hybrid_rerank(test, test_candidates_v2, test_chunk_maps_v2, test_questions_en)
print('Stage 1 dense-only test:', {k: v for k, v in finetuned_test.items() if k != 'per_query'})
print('Stage 3 dual+hybrid test:', {k: v for k, v in stage3_test.items() if k != 'per_query'})
print(f'Paired R@10 delta vs Stage 1 dense-only: {stage3_test["R@10"] - finetuned_test["R@10"]:+.4f}')

write_run(f'bge_m3_dual_hybrid_{RUN_SUFFIX}', BASE, True, stage3_test, test)
print()
print(f'Now run: python scripts/18_compare_runs.py --a results/runs/bge_m3_dual_hybrid_{RUN_SUFFIX}.json --b results/runs/bge_m3_finetuned_{RUN_SUFFIX}.json')


=== Stage 3: Dual-Query + Hybrid — Dev ===


Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00,  4.86it/s]


...

Batches:   0%|          | 0/617 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00,  5.05it/s]

Stage 1 dense-only test: {'R@1': 0.3055555555555556, 'R@5': 0.4166666666666667, 'R@10': 0.5, 'R@100': 0.6111111111111112, 'P@1': 0.3055555555555556, 'P@5': 0.10000000000000003, 'P@10': 0.05833333333333335, 'P@100': 0.009166666666666668, 'MRR@10': 0.3540454144620811, 'nDCG@10': np.float64(0.3879891879608068), 'slices': {'bengali_targets': {'n': 24, 'R@1': 0.4166666666666667, 'R@5': 0.5, 'R@10': 0.5416666666666666, 'R@100': 0.7083333333333334}, 'english_targets': {'n': 12, 'R@1': 0.08333333333333333, 'R@5': 0.25, 'R@10': 0.4166666666666667, 'R@100': 0.4166666666666667}}}
Stage 3 dual+hybrid test: {'R@1': 0.2222222222222222, 'R@5': 0.3611111111111111, 'R@10': 0.4444444444444444, 'R@100': 0.6111111111111112, 'MRR@10': 0.2840608465608466, 'nDCG@10': np.float64(0.3218743252831804)}
Paired R@10 delta vs Stage 1 dense-only: -0.0556
wrote /home/mlworkstation-admin/Shuvo/Dhara/results/runs/bge_m3_dual_hybrid_v6.json

Now run: python scripts/18_compare_runs.py --a results/runs/bge_m3_dual_hybrid_